# Phần Đức: Feature Engineering Luồng B và Decision Tree A/B

Notebook này dùng nguyên các tập `X_train_raw.csv`, `X_test_raw.csv`, `y_train.csv`, `y_test.csv` do Mạnh đã chia. Không chia lại dữ liệu.

Từ EDA, Luồng B giữ các nhóm có liên hệ rõ với Churn (hợp đồng, Internet, thanh toán, bảo mật/hỗ trợ, gia đình, tenure và chi phí), đồng thời tạo ba tương tác: `MonthToMonthFiber`, `MonthToMonthElectronicCheck`, `RiskServiceGap`. Việc điền thiếu và học mức one-hot chỉ thực hiện trên Train; Test chỉ được biến đổi bằng quy tắc đã học.

In [1]:
from pathlib import Path
import json
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src' / 'config.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / 'src' / 'config.py').exists()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print('Thư mục dự án:', PROJECT_ROOT)

Thư mục dự án: C:\Users\Admin\OneDrive\Documents\Machine Learning\BTL


In [2]:
from src.feature_engineering_b import main as tao_luong_b

tao_luong_b()

X_train_B: (5634, 29)
X_test_B : (1409, 29)
Da luu: C:\Users\Admin\OneDrive\Documents\Machine Learning\BTL\data\processed\X_train_B.csv
Da luu: C:\Users\Admin\OneDrive\Documents\Machine Learning\BTL\data\processed\X_test_B.csv
Da luu: C:\Users\Admin\OneDrive\Documents\Machine Learning\BTL\models\feature_engineer_B.pkl


In [3]:
import csv

def kich_thuoc_csv(path):
    with path.open('r', encoding='utf-8-sig', newline='') as handle:
        reader = csv.reader(handle)
        header = next(reader)
        row_count = sum(1 for _ in reader)
    return row_count, len(header)

processed = PROJECT_ROOT / 'data' / 'processed'
expected_rows = {'train': 5634, 'test': 1409}
for split, expected in expected_rows.items():
    raw_shape = kich_thuoc_csv(processed / f'X_{split}_raw.csv')
    b_shape = kich_thuoc_csv(processed / f'X_{split}_B.csv')
    y_shape = kich_thuoc_csv(processed / f'y_{split}.csv')
    assert raw_shape[0] == b_shape[0] == y_shape[0] == expected
    print(split, 'raw=', raw_shape, 'B=', b_shape, 'y=', y_shape, '— đạt')

train raw= (5634, 19) B= (5634, 29) y= (5634, 1) — đạt
test raw= (1409, 19) B= (1409, 29) y= (1409, 1) — đạt


## Tinh chỉnh Decision Tree

Hai mô hình dùng cùng `GridSearchCV` 5-fold có phân tầng trên Train, tối ưu F1 của lớp `Churn=Yes`. Lưới gồm `criterion`, `max_depth`, `min_samples_leaf` và `class_weight`. Tập Test chung chỉ được dùng một lần để báo cáo kết quả cuối.

In [4]:
from src.train_decision_tree import main as huan_luyen_decision_tree

huan_luyen_decision_tree()


===== DECISION TREE - LUONG A =====
Best CV F1 : 0.620934
Best params: {'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 8, 'min_samples_leaf': 40}
Accuracy   : 0.761533
Precision  : 0.535849
Recall     : 0.759358
F1-score   : 0.628319
CM [No,Yes]: [[789, 246], [90, 284]]
Da luu     : C:\Users\Admin\OneDrive\Documents\Machine Learning\BTL\results\duc_decision_tree_A.json



===== DECISION TREE - LUONG B =====
Best CV F1 : 0.626226
Best params: {'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': None, 'min_samples_leaf': 40}
Accuracy   : 0.745919
Precision  : 0.514184
Recall     : 0.775401
F1-score   : 0.618337
CM [No,Yes]: [[761, 274], [84, 290]]
Da luu     : C:\Users\Admin\OneDrive\Documents\Machine Learning\BTL\results\duc_decision_tree_B.json


In [5]:
for flow in ('A', 'B'):
    path = PROJECT_ROOT / 'results' / f'duc_decision_tree_{flow}.json'
    with path.open('r', encoding='utf-8') as handle:
        result = json.load(handle)
    print(f"Luồng {flow}:")
    print(json.dumps(result, ensure_ascii=False, indent=2))

Luồng A:
{
  "model_id": "decision_tree_A",
  "member": "Đức",
  "model": "Decision Tree",
  "flow": "A",
  "accuracy": 0.7615330021291696,
  "precision": 0.5358490566037736,
  "recall": 0.7593582887700535,
  "f1_score": 0.6283185840707964,
  "confusion_matrix": [
    [
      789,
      246
    ],
    [
      90,
      284
    ]
  ],
  "positive_label": "Yes",
  "cv_scoring": "f1 (Churn=Yes)",
  "cv_best_f1": 0.6209339280993824,
  "best_params": {
    "class_weight": "balanced",
    "criterion": "entropy",
    "max_depth": 8,
    "min_samples_leaf": 40
  },
  "feature_count": 45
}
Luồng B:
{
  "model_id": "decision_tree_B",
  "member": "Đức",
  "model": "Decision Tree",
  "flow": "B",
  "accuracy": 0.7459190915542938,
  "precision": 0.5141843971631206,
  "recall": 0.7754010695187166,
  "f1_score": 0.6183368869936035,
  "confusion_matrix": [
    [
      761,
      274
    ],
    [
      84,
      290
    ]
  ],
  "positive_label": "Yes",
  "cv_scoring": "f1 (Churn=Yes)",
  "cv_best_f1":

## Nhận xét

Luồng A có Accuracy, Precision và F1 trên Test cao hơn. Luồng B phát hiện được nhiều khách hàng rời bỏ hơn (Recall cao hơn) nhưng đổi lại có thêm false positive. Vì dữ liệu mất cân bằng và mục tiêu nghiệp vụ có thể ưu tiên không bỏ sót khách hàng sắp rời đi, cần báo cáo đầy đủ cả hai kết quả để Hùng chọn theo tiêu chí của nhóm.